<a href="https://colab.research.google.com/github/MaxRebella/CS539-Integration-Project/blob/main/P1_Data_Pipeline_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import statements:
# PyTorch. Building and training the CNN.
# TorchVision. Resizing, normalizing and augmenting the images.
# Datasets. Library for loading datasets (PlantVillage).
# Sklearn. Tools for evaluating our AI model.
# Matplotlib. Plotting and displaying our results.
# NumPy. Work with data.
# Huggingface. Import inorder to download PlantVillage.

import torch
import torchvision
from datasets import load_dataset
import sklearn
import matplotlib.pyplot
import numpy
from huggingface_hub import hf_hub_download
from zipfile import ZipFile
from pathlib import Path

In [2]:
# Download PlantVillage

download = hf_hub_download(
    repo_id="mohanty/PlantVillage",
    filename="data.zip",
    repo_type="dataset"
)

print("Dataset downloaded successfully!")

data.zip:   0%|          | 0.00/2.18G [00:00<?, ?B/s]

Dataset downloaded successfully!


In [3]:
# Data preperation. Split data into classes. Creates a folder with the classes. Splits the file names into the specific categories. Removes black & white images. Selects only the images belonging to our categories.

classes = {
    "Tomato___healthy": "healthy",
    "Tomato___Early_blight": "early_blight",
    "Tomato___Late_blight": "late_blight"
}

data_folder = Path("/content/tomato_data")

with ZipFile(download) as zip_file:
    for file in zip_file.namelist():
        parts = file.split("/")

        if "color" not in parts:
            continue

        # Find the disease category
        i = parts.index("color")

        if i + 2 >= len(parts):
            continue

        category = parts[i + 1]

        if category not in classes:
            continue

        if not file.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        # Determine where to save the image
        image_destination = data_folder.joinpath(classes[category], parts[-1])

        # Create the destination folder
        image_destination.parent.mkdir(parents=True, exist_ok=True)

        # Copy the image from the ZIP
        image_destination.write_bytes(zip_file.read(file))

print("Images extracted successfully!")


Images extracted successfully!


In [4]:
# Count images in each category

for folder in classes.values():
    images = list(data_folder.joinpath(folder).iterdir( ))
    print(folder, len(images))

healthy 1591
early_blight 1000
late_blight 1909


In [5]:
images = []
labels = []

for folder in classes.values():
    for image in sorted((data_folder / folder).iterdir( )):
        if image.suffix.lower() in (".jpg", ".jpeg",  ".png"):
            images.append(str(image))
            labels.append(folder)

print("Total images:", len(images))

Total images: 4500


In [6]:
# train_test_split will randomly split the dataset
from sklearn.model_selection import train_test_split

# Split 70/30. Training/Validation + Testing
train_images, remaining_images, train_labels, remaining_labels = train_test_split(
    images, labels,
    test_size= 0.30,
    random_state= 539,

    # Helps make sure there is roughly the same amount of each disease in each split.
    stratify= labels
)

# Second split: divide remaining images equally into development and testing
dev_images, test_images, dev_labels, test_labels = train_test_split(
    remaining_images, remaining_labels,
    test_size= 0.50,
    random_state= 539,
    stratify= remaining_labels
)

print("Training:", len(train_images))
print("Development:", len(dev_images))
print("Testing:", len(test_images))

print("Total:", len(train_images) + len(dev_images) + len(test_images))


Training: 3150
Development: 675
Testing: 675
Total: 4500


In [7]:
# Preprocessing.

from torchvision import transforms
from PIL import Image

image_size = 224
rotation_angle = 10
brightness_randomization_value = 0.1
contrast_randomization_value = 0.1

# Image Modifications only for training
train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(rotation_angle),
    transforms.ColorJitter(brightness= brightness_randomization_value, contrast= contrast_randomization_value),
    transforms.ToTensor(),

    # RGB value normalization
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

# Validation/Testing transformations only to normalize the image size and color.
eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
    ])

# Confirm preprocessing works
image = Image.open(train_images[0]).convert("RGB")
print(train_transform(image).shape)
print(eval_transform(image).shape)

torch.Size([3, 224, 224])
torch.Size([3, 224, 224])


In [8]:
# Create the dataset
from torch.utils.data import Dataset, DataLoader

label_ids = {
    "healthy": 0,
    "early_blight": 1,
    "late_blight": 2
}

class TomatoDataset(Dataset):

    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = Image.open(self.images[index]).convert("RGB")
        image = self.transform(image)
        label = label_ids[self.labels[index]]
        return image, label


# Create datasets
train_dataset = TomatoDataset(train_images, train_labels, train_transform)
dev_dataset = TomatoDataset(dev_images, dev_labels, eval_transform)
test_dataset = TomatoDataset(test_images, test_labels, eval_transform)


In [9]:
# DataLoader

batch_size = 32

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle= True
)

dev_loader = DataLoader(
    dev_dataset, batch_size=batch_size, shuffle= False
)

test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle= False
)

# Verify the DataLoader
images, labels = next(iter(train_loader) )

print("Image batch:", images.shape)
print("Label batch:", labels.shape)


Image batch: torch.Size([32, 3, 224, 224])
Label batch: torch.Size([32])


In [10]:
# Create the rudimentry CNN
import torch.nn

num_classes = 3
dropout_rate = 0.25

model = torch.nn.Sequential(

    torch.nn.Conv2d(3, 16, kernel_size= 5, padding= 2),
    torch.nn.BatchNorm2d(16),
    torch.nn.ReLU(),
    torch.nn.AvgPool2d(2),

    torch.nn.Conv2d(16, 32, kernel_size=5, padding= 2),
    torch.nn.BatchNorm2d(32),
    torch.nn.ReLU(),
    torch.nn.AvgPool2d(2),

    torch.nn.AdaptiveAvgPool2d((4, 4)),
    torch.nn.Flatten(),
    torch.nn.Dropout(dropout_rate),

    torch.nn.Linear(32 * 4 * 4, num_classes)
)


# Test the CNN with one batch
images, labels = next(iter(train_loader))
outputs = model(images)

print(model)
print(outputs.shape)

Sequential(
  (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU()
  (3): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (4): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (6): ReLU()
  (7): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (8): AdaptiveAvgPool2d(output_size=(4, 4))
  (9): Flatten(start_dim=1, end_dim=-1)
  (10): Dropout(p=0.25, inplace=False)
  (11): Linear(in_features=512, out_features=3, bias=True)
)
torch.Size([32, 3])


In [ ]:
# Train the CNN on Tomato Data

train_losses = []
train_accuracies = []

# Training Settings (CrossEntropy & Adam)
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 15

for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    average_loss = total_loss / total
    accuracy = 100 * correct / total
    train_losses.append(average_loss)
    train_accuracies.append(accuracy)

    print(f"Epoch {epoch+1} /{epochs}: Loss = {average_loss:.4f}, Accuracy = {accuracy:.2f}%")


In [ ]:
# Results plot
matplotlib.pyplot.figure(figsize=(10, 4))

matplotlib.pyplot.subplot(1, 2, 1)
matplotlib.pyplot.plot(range(1, epochs + 1), train_losses)
matplotlib.pyplot.xlabel("Epoch")
matplotlib.pyplot.ylabel("Loss")
matplotlib.pyplot.title("Training Loss")

matplotlib.pyplot.subplot(1, 2, 2)
matplotlib.pyplot.plot(range(1, epochs + 1), train_accuracies)
matplotlib.pyplot.xlabel("Epoch")
matplotlib.pyplot.ylabel("Accuracy (%)")
matplotlib.pyplot.title("Training Accuracy")

matplotlib.pyplot.tight_layout()
matplotlib.pyplot.show()

# Save the Model
torch.save(model.state_dict(), "tomato_cnn.pth")
print("Model saved! :)")